# Људски надзор: Прелазни бодови пре акције, рангирање ризика и евиденција ревизије

README за ову лекцију уводи Људски надзор кратким фрагментом који од корисника тражи да `ОДОБРИ` или `ОДБИЈЕ` након што агент већ произведе одговор. Тај образац је добар почетак, али у продукцијском HITL-у су обично потребна још три додатна елемента:

1. **прелазни бод пре акције** који се покреће **пре** него агент изврши ризичан корак, како би трошкови, неисправљивост и кашњење били под контролом.
2. **рангирање ризика**, тако да се акције ниског ризика аутоматски извршавају, средње ризичне акције одобравају у серији, а само акције високог ризика блокирају људска интервенција.
3. **дневник ревизије и круг ревизије**, тако да се свака одлука са прелазног бода записује као JSONL, а одбијање поново покреће агента са структурираном разлогом уместо само исписивања `Revising...`.

Овај нотебоок гради сваки од ових елемената на бази истих примитива као `06-system-message-framework.ipynb`. Извршава се од почетка до краја у режиму `DEMO_MODE = True` (није потребан интерактивни унос) или уз стварне `input()` уносе када је `DEMO_MODE = False`. Напомена: у режиму DEMO_MODE поновни покушај из трећег циља је скриптован тако да су механике циклуса видљиве од почетка до краја. Стварна ревизијом вођена рекласификација захтева `DEMO_MODE = False` и оператера.

**Изван опсега (решава се у другим лекцијама):** аутентикација и контрола приступа (Лекција 06 README претња 2), посредник позива алата (Лекција 14 дубока анализа MAF), обрасци дебате више агената.


In [ ]:
import json
import os
from datetime import datetime, timezone
from pathlib import Path

from dotenv import load_dotenv
from azure.identity import DefaultAzureCredential, get_bearer_token_provider
from openai import OpenAI

load_dotenv()

DEMO_MODE = True  # set False to use real input() prompts

# Per-run unique log filename so demo runs don't overwrite each other and
# the notebook doesn't touch any pre-existing gate_log.jsonl in the working
# directory.
GATE_LOG_PATH = Path(
    f"gate_log_{datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%SZ')}.jsonl"
)

# This notebook uses the Azure OpenAI Responses API via the stable /openai/v1/ endpoint.
# GitHub Models is deprecated (retiring July 2026) and does not support the Responses API.
endpoint = os.environ.get("AZURE_OPENAI_ENDPOINT", "")
if not endpoint:
    raise RuntimeError(
        "AZURE_OPENAI_ENDPOINT environment variable is not set. This notebook needs "
        "an Azure OpenAI resource with a model deployment that supports the Responses "
        "API. Set AZURE_OPENAI_ENDPOINT and AZURE_OPENAI_DEPLOYMENT in "
        "your environment or a local .env file, then run `az login`."
    )

deployment = os.environ["AZURE_OPENAI_DEPLOYMENT"]

# Authenticate with Entra ID (run `az login` first). No api_version is needed.
token_provider = get_bearer_token_provider(
    DefaultAzureCredential(),
    "https://cognitiveservices.azure.com/.default",
)

client = OpenAI(
    base_url=f"{endpoint.rstrip('/')}/openai/v1/",
    api_key=token_provider,
)



## Шаблон 1: Претходна провера акције

ХИТЛ фрагмент из README-а прво позива агента, а затим тражи од корисника да одобри резултат. То је **пост-акциона** процедура. Агент је већ извршио, па је трошак позива LLM већ плаћен, и свака споредна последица (послата имејл порука, упис у базу података, објављени коментар) се већ десила.

**Претходна** процедура убацује проверу пре него што агент изврши ризичан корак. Агент предлаже акцију, провера одлучује да ли да изврши, и само уз одобрење долази до споредне последице.

| Аспект | Пост-акционо одобрење (фрагмент из README) | Претходна провера (овај нотебук) |
|---|---|---|
| Када се извршава одобрење? | Након што је агент произвео резултат | Пре извршења било које споредне последице |
| Трошак LLM при одбијању | Већ плаћен | Плаћа се само за предлог, не за акцију |
| Неопозиве споредне последице | Могуће (акција је већ извршена) | Спречене |
| Јасноћа ревизије | Одобрење као испис у конзоли | Одобрење као JSON запис са временском ознаком, акцијом и разлогом |


In [ ]:
def gate_action(action_description: str, risk_tier: str, attempt: int = 0) -> dict:
    """Run a single pre-action gate.

    Returns a decision dict with keys: decision, reason, ts.
    Decision is one of: approve, deny, escalate.
    Safe default on EOF or unexpected input is deny.

    DEMO_MODE behavior: high-risk actions are denied on attempt 0 and
    auto-approved on attempt >= 1. This is scripted approval to show the
    loop mechanics (deny -> retry -> approve). It is NOT revision-driven
    re-classification. Real revision-driven re-classification requires
    DEMO_MODE=False and a human operator who evaluates the revised
    proposal on its own merits.
    """
    print(f"[gate] proposed action ({risk_tier}, attempt={attempt}): {action_description}")

    if DEMO_MODE:
        if risk_tier == "high":
            decision = "approve" if attempt >= 1 else "deny"
            reason = (
                "DEMO_MODE: scripted approval on retry to show loop mechanics"
                if attempt >= 1
                else "DEMO_MODE: high risk denied on first attempt"
            )
        else:
            decision = "approve"
            reason = f"DEMO_MODE canned response for tier={risk_tier}"
    else:
        try:
            raw = input("[gate] approve / deny / escalate? ").strip().lower()
        except EOFError:
            raw = ""
        if raw in {"approve", "deny", "escalate"}:
            decision, reason = raw, "operator input"
        elif raw == "":
            decision, reason = "deny", "no input received, defaulted to deny"
        else:
            decision, reason = "deny", f"invalid input {raw!r}, defaulted to deny"

    return {
        "decision": decision,
        "reason": reason,
        "action": action_description,
        "risk_tier": risk_tier,
        "ts": datetime.now(timezone.utc).isoformat(),
    }


## Образац 2: Рангирање ризика

Ни свака акција не захтева људско одобрење. Претраживање само за читање преко јавног API-ја има другачије улоге него слање мејла купцу. Постављање оба у исту категорију троши пажњу оператера и успорава агента.

Једноставан модел са 3 нивоа:

| Ниво | Примери | Ток одобрења |
|---|---|---|
| `low` (само за читање) | Претрага базе знања, претрага опција летова, преузимање јавне веб странице | Аутоматско извршавање, евидентирано за ревизију |
| `medium` (јефтино мењање) | Кеширање резултата, увећање бројача, заказивање подсетника | Аутоматско извршавање, али са свакодневним прегледом у пакетима |
| `high` (спољна или неповратна акција) | Слање мејла, наплата картице, објава на јавном каналу | Блокирано док људски не буде одобрено |

Ово је једно рангирање. Продуктивни системи често користе прецизније нивое (нпр. AWS IAM нивои дозвола, рангирање засновано на улогама). Верује се да је ова трониво верзија најмања корисна верзија за агента који меша акције само за читање и оне које доводе до последица.

Класификатор испод користи кључне речи као хеуристику да би демонстрација остала детерминистичка и јефтина. У продукционом систему заменили бисте ову опцију класификатором са машинским учењем или системом политика.


In [ ]:
LOW_RISK_KEYWORDS = {
    "look", "lookup", "search", "fetch", "read", "query", "view",
    "get", "list", "weather", "summarize",
}
HIGH_RISK_KEYWORDS = {
    "send", "email", "post", "publish", "charge", "pay", "transfer",
    "delete", "drop", "cancel", "refund",
}
MEDIUM_RISK_KEYWORDS = {
    "cache", "schedule", "reminder", "book", "reserve", "update",
    "increment", "log",
}

AUTO_APPROVE_REASONS = {
    "low": "auto-approved (low risk)",
    "medium": "auto-approved (medium risk, queued for batched review)",
}


def classify_risk(action: str) -> str:
    """Classify an action string into one of: low, medium, high.

    Keyword-based heuristic. Checks high-risk first (most severe), then
    low-risk explicit reads, then medium-risk mutations. Unrecognized
    actions default to medium, not low.

    Default for unrecognized actions is 'medium', not 'low'. A read-only
    keyword set will always have blind spots, and the parent README's
    threat list (critical-system access, knowledge-base poisoning,
    cascading errors) all involve cases an action-name alone cannot rule
    out. Routing unknown actions through batched review is the safer
    default than auto-executing them.
    """
    text = action.lower()
    if any(kw in text for kw in HIGH_RISK_KEYWORDS):
        return "high"
    if any(kw in text for kw in LOW_RISK_KEYWORDS):
        return "low"
    if any(kw in text for kw in MEDIUM_RISK_KEYWORDS):
        return "medium"
    # Explicit fail-safe default: unrecognized actions route to batched review.
    return "medium"


def tiered_gate(action: str, attempt: int = 0) -> dict:
    """Classify then gate. Low and medium tiers auto-approve; high blocks."""
    tier = classify_risk(action)
    if tier in AUTO_APPROVE_REASONS:
        return {
            "decision": "approve",
            "reason": AUTO_APPROVE_REASONS[tier],
            "action": action,
            "risk_tier": tier,
            "ts": datetime.now(timezone.utc).isoformat(),
        }
    return gate_action(action, tier, attempt=attempt)


## Образац 3: Дневник ревизије и циклус ревизија

`print("Response approved.")` није дневник ревизије. За поузданост, свака одлука о пролазу треба да буде забележена као структуирани догађај који касније можете упитати, репродуковати или приложити прегледу инцидента.

Два дела:

1. **JSONL само за додавање.** Један ред по одлуци, са временском ознаком, акцијом, нивоом, одлуком, разлогом. Лако за претрагу, лако се касније шаље у прави дневник.
2. **Циклус ревизије при одбијању.** Када капија врати `deny`, агент сам себи поново поставља питање са разлогом одбијања у контексту, тако да следећи предлог може да избегне проблем.


In [ ]:
def log_decision(decision: dict) -> None:
    """Append a gate decision to the JSONL audit log."""
    with GATE_LOG_PATH.open("a", encoding="utf-8") as f:
        f.write(json.dumps(decision) + "\n")


def propose_action(goal: str, prior_rejection: str | None = None) -> str:
    """Ask the LLM to propose a concrete next action for a goal.

    If prior_rejection is provided, it is fed back so the LLM can avoid
    the same failure mode in the next proposal.
    """
    system = (
        "You are an action planner for an agent. Propose ONE concrete next\n"
        "action (a single sentence) toward the user's goal. If a prior\n"
        "rejection reason is given, propose a different action that addresses\n"
        "the rejection."
    )
    user_text = f"Goal: {goal}"
    if prior_rejection:
        user_text += f"\n\nPrior proposal was denied. Reason: {prior_rejection}"

    response = client.responses.create(
        model=deployment,
        input=[
            {"role": "system", "content": system},
            {"role": "user", "content": user_text},
        ],
        store=False,
    )
    return response.output_text.strip()


def run_with_revision(goal: str, max_revisions: int = 2) -> dict:
    """Propose, gate, and on rejection revise up to max_revisions times."""
    prior_reason: str | None = None
    for attempt in range(max_revisions + 1):
        action = propose_action(goal, prior_rejection=prior_reason)
        decision = tiered_gate(action, attempt=attempt)
        decision["attempt"] = attempt
        log_decision(decision)
        if decision["decision"] == "approve":
            return decision
        prior_reason = decision["reason"]
    return {**decision, "final": "max_revisions_reached"}



In [ ]:
# End-to-end demo: three goals at three different risk profiles.
# GATE_LOG_PATH is per-run (timestamped) so no prior log is touched.

goals = [
    "Look up the weather in Seattle for the customer's trip planning.",
    "Schedule a reminder for the customer to check in 24 hours before their flight.",
    "Send a marketing email to the customer about premium upgrade options.",
]

for goal in goals:
    print(f"\n=== Goal: {goal} ===")
    outcome = run_with_revision(goal, max_revisions=1)
    print(f"[final] {outcome['decision']} ({outcome['reason']})")

print(f"\n=== Audit log ({GATE_LOG_PATH.name}) ===")
for line in GATE_LOG_PATH.read_text(encoding="utf-8").splitlines():
    record = json.loads(line)
    print(f"  [{record['risk_tier']:6s}] {record['decision']:8s} "
          f"attempt={record.get('attempt', '?')} action={record['action'][:140]}")


## Додатни ресурси

Још неколико јавних пројеката имплементира варијације ових HITL образаца. Упоредите приступе да бисте пронашли шта одговара вашем стеку:

- **LangChain** алатке за људски у петљи (HITL) омотаче ([документација](https://python.langchain.com/docs/integrations/tools/human_tools)): омотачи алатки који паузирају извршавање за унос од стране човека.
- **AutoGen** `UserProxyAgent` ([v0.2 документација](https://microsoft.github.io/autogen/0.2/docs/topics/human-in-the-loop); AutoGen v0.4+ је реструктурисан): користи улогу агента посебно да представља човека у мулти-агентским разговорима.
- **Microsoft Agent Framework (MAF)** middleware за позив функција ([документација](https://learn.microsoft.com/agent-framework/)): middleware који ради око сваког позива алатке/функције, погодан за контролу логике и токове одобравања.

Сваки пројекат другачије рукује три подобразаца: LangChain их омотава као алатке, AutoGen користи улогу агента, а Microsoft Agent Framework користи middleware за позив функција. Процитати једну или две имплементације од почетка до краја пре него што одаберете дизајн за свог агента.


---

<!-- CO-OP TRANSLATOR DISCLAIMER START -->
**Изјава о одрицању одговорности**:
Овај документ је преведен коришћењем услуге за аутоматски превод [Co-op Translator](https://github.com/Azure/co-op-translator). Иако тежимо тачности, имајте у виду да аутоматски преводи могу садржати грешке или нетачности. Оригинални документ на његовом изворном језику треба сматрати ауторитативним извором. За критичне информације препоручује се професионални људски превод. Нисмо одговорни за било каква неспоразума или погрешна тумачења која произилазе из коришћења овог превода.
<!-- CO-OP TRANSLATOR DISCLAIMER END -->
